# Phase 2: Feature Engineering & Merging Exogenous Signals
**Objective:** We will merge the core sales data with exogenous store and weather features. Then, we will engineer temporal indicators (Week, Month) and handle the missing promotional markdowns so the LSTM model has dense, rich signals to learn from.


In [8]:
import pandas as pd
import numpy as np

# 1. Emulating your data_loader.py logic
train = pd.read_csv("../data/train.csv")
features = pd.read_csv("../data/features.csv")
stores = pd.read_csv("../data/stores.csv")

# Merge exactly as you structured it
df = train.merge(features, on=["Store", "Date"], how="left")
df = df.merge(stores, on="Store", how="left")

# Convert Date
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date')

print(f"Merged Dataset Shape: {df.shape}")

Merged Dataset Shape: (421570, 17)


## Engineering Temporal & Holiday Signals
To help the models (specifically LSTM) recognize seasonal trends, we extract the week and month. We also need to handle the `MarkDown` columns which contain thousands of NaNs where no promotions were running.

In [9]:
# Extract temporal features
df['Week'] = df['Date'].dt.isocalendar().week
df['Month'] = df['Date'].dt.month
df['Year'] = df['Date'].dt.year

# Fill missing markdowns with 0 (No promotion running)
markdown_cols = ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']
df[markdown_cols] = df[markdown_cols].fillna(0)

# Fill missing CPI and Unemployment with the previous valid value (forward fill)
df['CPI'] = df['CPI'].ffill()
df['Unemployment'] = df['Unemployment'].ffill()

# --- THE BULLETPROOF HOLIDAY FIX ---
# Find all columns that have "holiday" in the name (case-insensitive)
holiday_cols = [col for col in df.columns if 'holiday' in col.lower()]

if holiday_cols:
    target_col = holiday_cols[0] # Grab the first one it finds (e.g., IsHoliday_x)
    holiday_data = df[target_col].astype(int) # Convert true/false to 1/0
    
    # Drop all the messy original columns
    df = df.drop(columns=holiday_cols)
    
    # Create one clean, unified column
    df['Is_Holiday'] = holiday_data
else:
    print("WARNING: Could not find any holiday column. Available columns are:")
    print(df.columns)

# Display the engineered dataset
df.head()

,Store,Dept,Date,Weekly_Sales,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,Type,Size,Week,Month,Year,Is_Holiday
0,1,1,2010-02-05,24924.50,42.31,2.572,0.0,0.0,0.0,0.0,0.0,211.096358,8.106,A,151315,5,2,2010,0
277665,29,5,2010-02-05,15552.08,24.36,2.788,0.0,0.0,0.0,0.0,0.0,131.527903,10.064,B,93638,5,2,2010,0
277808,29,6,2010-02-05,3200.22,24.36,2.788,0.0,0.0,0.0,0.0,0.0,131.527903,10.064,B,93638,5,2,2010,0
277951,29,7,2010-02-05,10820.05,24.36,2.788,0.0,0.0,0.0,0.0,0.0,131.527903,10.064,B,93638,5,2,2010,0
278094,29,8,2010-02-05,20055.64,24.36,2.788,0.0,0.0,0.0,0.0,0.0,131.527903,10.064,B,93638,5,2,2010,0
